In [1]:
import os
os.environ.pop("DATABRICKS_CLUSTER_ID", None)
os.environ["DATABRICKS_SERVERLESS_COMPUTE_ID"] = "auto"

from databricks.connect import DatabricksSession

spark = DatabricksSession.builder.getOrCreate()
spark.sql("USE CATALOG sokoni_retail")
base = "abfss://sokoni@sokoniadls.dfs.core.windows.net/bronze_layer/"

In [3]:
spark.read.option("header", True).csv(
    "abfss://sokoni@sokoniadls.dfs.core.windows.net/bronze_layer/products/"
).show(10, truncate=False)

+----------+------------------------------------+-------------+--------------+---------+---------+-----------------+
|product_id|product_name                        |category     |brand         |unit     |pack_size|current_price_kes|
+----------+------------------------------------+-------------+--------------+---------+---------+-----------------+
|P0001     |House Brand Porridge Flour (Regular)|Staples      |House Brand   |500g     |Regular  |90               |
|P0002     |Kibo Bleach (Regular)               |Household    |Kibo          |1L       |Regular  |155              |
|P0003     |Kibo Mandazi (Regular)              |Bakery       |Kibo          |pack of 6|Regular  |90               |
|P0004     |Kenblest Oats (Small)               |Staples      |Kenblest      |500g     |Small    |135              |
|P0005     |Tusker Fresh Sanitary Pads (Family) |Personal Care|Tusker Fresh  |10 pack  |Family   |300              |
|P0006     |Golden Harvest Yoghurt (Regular)    |Dairy        |G

In [4]:
spark.table("sokoni_retail.silver.sales_rejected") \
     .groupBy("reject_reason").count().show()

+----------------+-----+
|   reject_reason|count|
+----------------+-----+
|invalid_quantity|19588|
+----------------+-----+



In [2]:
p = spark.table("sokoni_retail.gold.dim_product")
s = spark.table("sokoni_retail.silver.sales_clean")

print("sales with no product match:",
      s.join(p, "product_id", "left_anti").count())

sales with no product match: 0
